In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from sklearn.metrics import mean_absolute_error, mean_squared_error
from models import *
from plots import *

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = load_data()

In [ ]:
data.head()

In [ ]:
target = 't_seasdiff'
exog=['lagged_tmed_24', 'lagged_prec_24', 'lagged_tmin_24', 'lagged_tmax_24']

In [ ]:
data.head()

In [ ]:
if target == 't_seasdiff':
    data["t_seasdiff"]= data["tdiff"].diff(24)
    data = data.dropna(subset=["t_seasdiff"])

# SARIMA / SARIMAX

In [ ]:
len(data)

In [ ]:
p, d, q = 0, 1, 1
P, D, Q, s = 1, 0, 0, 12


In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s, exog= exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
from utils import sarima_candidates_from_acf

cand_res = sarima_candidates_from_acf(data[target], m=12, max_p=3, max_q=2, max_P=3, max_Q=3,d=0, D=0, max_lag=50)

candidates = cand_res["candidates"]
for i, (p, d, q, P, D, Q, s) in enumerate(candidates, 1):
    print(f"{i:02d}. SARIMA({p},{d},{q})({P},{D},{Q},{s})")

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=None,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=exog,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

## Using AutoSarima with the current train/test split

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=24, no_windows=10, m=12, exog=exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
best_info, scores_df = select_best_sarima_cv(
    df=data,
    target=target,
    candidates=candidates,
    train_years=8,
    forecast_months=24,
    test_start_year=1995,
    exog=None,
    metric="rmse",
    use="avg",
    verbose=True,
)

cv = best_info["cv_results"]



In [ ]:
from models import _plot_lstm_cv_results


cv = best_info["cv_results"]  


_plot_lstm_cv_results(
    df=data,
    target=target,
    split_metrics=cv.get("split_metrics", []),  
    forecasts=cv.get("forecasts", []),
    actuals=cv.get("actuals", []),
    forecast_months=24,
    residuals=cv.get("residuals", []),
    lb=pd.DataFrame(cv.get("lb_results", [])),
)

In [ ]:
last_fold = best_info["cv_results"]["folds"][-1]
plot_last_fold(last_fold, acf_lags=12, lb_lags=1)

In [ ]:
best_info, scores_df = select_best_sarima_cv(
    df=data,
    target=target,
    candidates=candidates,
    train_years=8,
    forecast_months=24,
    test_start_year=1995,
    exog=exog,
    metric="rmse",
    use="overall",
    verbose=True,
)



In [ ]:
cv = best_info["cv_results"]  

plot_sarima_cv_results(
    df=data,
    target=target,
    split_metrics=cv.get("split_metrics", []),  
    forecasts=cv.get("forecasts", []),
    actuals=cv.get("actuals", []),
    forecast_months=24,
    residuals=cv.get("residuals", []),
    lb=pd.DataFrame(cv.get("lb_results", [])),
)

# LSTMs

In [ ]:
res = lstm_grid_search_cv(data, 'tdiff', train_years=8, forecast_months=24, test_start_year=1996, epochs=60, verbose=0, param_grid={
    'exog': [['lagged_tmed', 'lagged_tmin', 'lagged_prec', 'lagged_tmax'], ['lagged_tmin', 'lagged_prec', 'lagged_tmax'], ['lagged_tmed', 'lagged_prec'], ['lagged_tmin', 'lagged_prec', 'lagged_tmax']],
    'lr': [0.001],
    'hidden_size': [50, 25, 100],
    'num_layers': [1, 2],
    'dropout': [0.1, 0.2, 0.3]
    
})

In [ ]:
print_grid_search_results(res['results'], res['best_overall_score'], res['best_overall_results'], res['best_overall_params'],res['best_avg_score'], res['best_avg_results'], res['best_avg_params'],res['best_last_fold_score'], res['best_last_fold_results'], res['best_last_fold_params'])

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=['lagged_tmed_24', 'lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24', 'lagged_tdiff_24'],  
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=100,
    num_layers=1,
    dropout=0.2,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=['lagged_tmed_12', 'lagged_tmin_12', 'lagged_prec_12', 'lagged_tmax_12', 'lagged_tdiff_12'], 
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=25,
    num_layers=2,
    dropout=0.2,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=1996,
    exog=['lagged_tmed_1', 'lagged_tmin_1', 'lagged_prec_1', 'lagged_tmax_1', 'lagged_tdiff_1', 'lagged_tmed_12', 'lagged_tmin_12', 'lagged_prec_12', 'lagged_tmax_12', 'lagged_tdiff_12' ], 
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=25,
    num_layers=2,
    dropout=0.2,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=2010,
    exog=['lagged_tmed', 'lagged_tmin', 'lagged_prec', 'lagged_tmax', 'lagged_tdiff'], 
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=25,
    num_layers=2,
    dropout=0.2,
    plot=True,
)

In [ ]:
results = lstm_time_series_cv(
    df=data,
    target='tdiff',
    train_years=8,
    forecast_months=24,
    test_start_year=2010,
    exog=['lagged_tmin_24', 'lagged_prec_24', 'lagged_tmax_24', 'lagged_tdiff_24'],  
    seq_length=24,
    epochs=60,
    batch_size=32,
    lr=0.001,
    hidden_size=100,
    num_layers=1,
    dropout=0.2,
    plot=True,
)